In [3]:
import pandas as pd
elo_rating = pd.read_csv('../data/raw/elo_ratings_wc2026.csv')
results = pd.read_csv('../data/raw/results.csv')
results['date']=pd.to_datetime(results['date'])
elo_rating['snapshot_date']=pd.to_datetime(elo_rating['snapshot_date'])
print("results:", results.shape)
print("elo_rating:", elo_rating.shape)


results: (49477, 9)
elo_rating: (4683, 23)


In [4]:
print(results.columns.tolist())
print(elo_rating.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
['year', 'snapshot_date', 'country', 'rank', 'country_code', 'rating', 'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host']


In [6]:
results['year'] = results['date'].dt.year
print(results[['date', 'year', 'home_team', 'away_team']].head(5))

        date  year home_team away_team
0 1872-11-30  1872  Scotland   England
1 1873-03-08  1873   England  Scotland
2 1874-03-07  1874  Scotland   England
3 1875-03-06  1875   England  Scotland
4 1876-03-04  1876  Scotland   England


In [8]:
elo_rating = elo_rating[elo_rating['snapshot_date'] < '2026-06-28']
print(elo_rating.shape)

(4635, 23)


In [11]:
print(elo_rating['year'].dtype)
print(elo_rating['year'].head())
print(results['year'].dtype)
print(results['year'].head())

int64
0    1901
1    1901
2    1902
3    1902
4    1902
Name: year, dtype: int64
int32
0    1872
1    1873
2    1874
3    1875
4    1876
Name: year, dtype: int32


In [12]:
results['year'] = results['year'].astype(int)
print(results['year'].dtype)

int64


In [18]:
matches_with_elo_home=pd.merge(results, elo_rating, left_on=['home_team', 'year'], right_on=['country', 'year'], how='left')
print(matches_with_elo_home.shape)
print(matches_with_elo_home.columns.to_list())
print(matches_with_elo_home.head(3))

(49477, 32)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country_x', 'neutral', 'year', 'snapshot_date', 'country_y', 'rank', 'country_code', 'rating', 'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host']
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   

  country_x  neutral  year  ... matches_home matches_away  matches_neutral  \
0  Scotland    False  1872  ...          NaN          NaN              NaN   
1   England    False  1873  ...          NaN          NaN              NaN   
2  Scotland    False 

In [19]:
matches_with_elo_home = matches_with_elo_home[['date', 'home_team', 'away_team', 
                                                'home_score', 'away_score', 
                                                'tournament', 'neutral', 
                                                'year', 'rating']]

matches_with_elo_home = matches_with_elo_home.rename(columns={'rating': 'home_elo'})
print(matches_with_elo_home.shape)
print(matches_with_elo_home.head(3))

(49477, 9)
        date home_team away_team  home_score  away_score tournament  neutral  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly    False   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly    False   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly    False   

   year  home_elo  
0  1872       NaN  
1  1873       NaN  
2  1874       NaN  


In [23]:
matches_with_elo = pd.merge(matches_with_elo_home, elo_rating,
                             left_on=['away_team', 'year'],
                             right_on=['country', 'year'],
                             how='left')

matches_with_elo = matches_with_elo[['date', 'home_team', 'away_team',
                                      'home_score', 'away_score',
                                      'tournament', 'neutral',
                                      'year', 'home_elo', 'rating']]

matches_with_elo = matches_with_elo.rename(columns={'rating': 'away_elo'})

print(matches_with_elo.shape)
print(matches_with_elo.head(3))

(49477, 10)
        date home_team away_team  home_score  away_score tournament  neutral  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly    False   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly    False   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly    False   

   year  home_elo  away_elo  
0  1872       NaN       NaN  
1  1873       NaN       NaN  
2  1874       NaN       NaN  


In [24]:
print(matches_with_elo[matches_with_elo['year'] >= 2020].head(3))

            date home_team away_team  home_score  away_score tournament  \
43375 2020-01-07  Barbados    Canada         1.0         4.0   Friendly   
43376 2020-01-09   Moldova    Sweden         0.0         1.0   Friendly   
43377 2020-01-10  Barbados    Canada         1.0         4.0   Friendly   

       neutral  year  home_elo  away_elo  
43375     True  2020       NaN    1599.0  
43376     True  2020       NaN    1820.0  
43377     True  2020       NaN    1599.0  


In [27]:
matches_with_elo= matches_with_elo[ (matches_with_elo['home_elo'].notna()) & (matches_with_elo['away_elo'].notna()) ]
print(matches_with_elo.shape)

(7429, 10)


In [29]:
matches_with_elo.to_csv('../data/processed/matches_with_elo.csv',index=False)